In [13]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import requests
from bs4 import BeautifulSoup 
import numpy as np

# P3 Limpieza csv

In [ ]:

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

df = pd.read_csv('../data/corpus_v1.csv')

def limpiar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r'[^a-z\s]', '', texto)
    words = texto.split()
    words = [w for w in words if w not in stop_words and len(w) > 2]
    return " ".join(words)

df['texto_limpio'] = df['texto'].apply(limpiar_texto)
vocabulario_total = set(" ".join(df['texto_limpio']).split())
print(f"Tamaño del vocabulario final: {len(vocabulario_total)} palabras únicas.")

df.to_csv('../data/corpus_limpio.csv', index=False)

Tamaño del vocabulario final: 6877 palabras únicas.


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nicom\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:

df = pd.read_csv('../data/corpus_limpio.csv')
filas_vacias = df['texto_limpio'].isna()
for index, row in df[filas_vacias].iterrows():
    url = row['url']
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        respuesta = requests.get(url, headers=headers, timeout=10)
        if respuesta.status_code == 200:
            soup = BeautifulSoup(respuesta.text, 'html.parser')
            parrafos = soup.find_all('p')
            texto_extraido = " ".join([p.get_text() for p in parrafos])
            if len(texto_extraido) > 50:
                df.at[index, 'texto_limpio'] = texto_extraido
                print(f"Recuperado: {url}")
            else:
                print(f"La página cargó, pero no se encontró texto en: {url}")
        else:
            print(f"Error {respuesta.status_code} al acceder a: {url}")
    except Exception as e:
        print(f"Fallo de conexión con {url}: {e}")

df.to_csv('../data/corpus_recuperado.csv', index=False)

Recuperado: https://investors.palantir.com/news-details/2026/Palantir-Joins-U-S--Army-and-Industry-Partners-for-Right-to-Integrate-Hackathon-Sprint-for-Defense-wide-Interoperability/
Recuperado: https://investors.palantir.com/news-details/2026/GE-Aerospace-and-Palantir-Expand-Partnership-to-Transform-Military-Aircraft-Readiness-with-AI/
La página cargó, pero no se encontró texto en: https://www.palantir.com/q1-2026-letter/en/
La página cargó, pero no se encontró texto en: https://www.palantir.com/newsroom/letters/aip-in-action/en/
La página cargó, pero no se encontró texto en: https://www.palantir.com/q4-2025-letter/en/
La página cargó, pero no se encontró texto en: https://www.palantir.com/newsroom/press-releases/palantir-and-oracle-announce-strategic-partnership/
La página cargó, pero no se encontró texto en: https://www.palantir.com/newsroom/press-releases/palantir-and-panasonic-energy-expand-partnership/
La página cargó, pero no se encontró texto en: https://www.palantir.com/newsro

# P4 Creacion Matriz

In [ ]:
df = pd.read_csv('../data/corpus_recuperado.csv')
df_valido = df.dropna(subset=['texto_limpio']).copy()
vectorizador = TfidfVectorizer()
X = vectorizador.fit_transform(df_valido['texto_limpio'])
A_sparse = X.T
A = A_sparse.toarray()
m, n = A.shape
vocabulario = vectorizador.get_feature_names_out()
print(f"Dimensiones de la matriz A: {m} (terminos) x {n} (documentos)")

Dimensiones de la matriz TF-IDF: (41, 6970)
Dimensiones de la matriz A: 6970 (terminos) x 41 (documentos)


# P5 Calculo de la SVD


In [ ]:
U, S, Vt = np.linalg.svd(A, full_matrices=False)
Sigma = np.diag(S)
print(f"Dimensiones de U: {U.shape}, Sigma: {Sigma.shape}, Vt: {Vt.shape}")